<a href="https://colab.research.google.com/github/Sirmj-1986/JMC_Minerals/blob/main/SAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import ee
import geemap

In [11]:
ee.Authenticate()

True

In [12]:
ee.Initialize(
    project = 'ee-mjfati0086')

In [13]:
kogi = ee.Geometry.Polygon([
    [6.384633300164806, 8.00523783856405],
    [6.384633300164806, 7.918874512340253],
    [6.588567015985118, 7.918874512340253],
    [6.588567015985118, 8.00523783856405]
])

In [14]:
center_lat = (8.00523783856405 + 7.918874512340253) / 2.0
center_lon = (6.384633300164806 + 6.588567015985118) / 2.0
Map = geemap.Map(center=(center_lat, center_lon), zoom=12)

In [15]:
Map.addLayer(kogi, {}, 'ROI')
Map.centerObject(kogi, 13)

Map

Map(center=[7.962067105113433, 6.486600158075314], controls=(WidgetControl(options=['position', 'transparent_b…

In [16]:
aster_col = (
    ee.ImageCollection("ASTER/AST_L1T_003")
    .filterBounds(kogi)
    .filter(ee.Filter.lt('CLOUDCOVER', 0.1))
    .filter(ee.Filter.calendarRange(11, 3, 'month'))
)

asterRaw = aster_col.median().clip(kogi)

asterBandNames = ['B01','B02','B3N','B04','B05','B06','B07','B08','B09']
asterBands = asterRaw.select(asterBandNames)

In [17]:
Map.addLayer(asterRaw, {}, 'Raw ASTER', shown=False)
Map

Map(center=[7.962067105113433, 6.486600158075314], controls=(WidgetControl(options=['position', 'transparent_b…

In [18]:
# Dark Object Subtraction (DOS)
def apply_dos(image, band_names, geometry, scale=30):
    """
    image: ee.Image
    band_names: list of band names (strings)
    geometry: ee.Geometry for statistics
    """
    corrected_bands = []
    for b in band_names:
        band = image.select(b)

        dark_val = band.reduceRegion(
            reducer=ee.Reducer.percentile([1]),
            geometry=geometry,
            scale=scale,
            maxPixels=1e9
        ).get(b)
        dark_num = ee.Number(dark_val)
        corrected = band.subtract(dark_num).rename(b)

        corrected_bands.append(corrected.max(0))
    return ee.Image.cat(corrected_bands)

asterDOS = apply_dos(asterRaw, asterBandNames, kogi, scale=30).clip(kogi)
Map.addLayer(asterDOS, {}, 'ASTER DOS Corrected', shown=False)
Map


Map(center=[7.962067105113433, 6.486600158075314], controls=(WidgetControl(options=['position', 'transparent_b…

In [19]:
# SAM function and apply to references
references = {
    "Calcite":    [0.19, 0.22, 0.23, 0.28, 0.32, 0.31, 0.29, 0.26, 0.22],
    "Kaolinite":  [0.12, 0.14, 0.16, 0.20, 0.24, 0.23, 0.22, 0.20, 0.18],
    "IronOxide":  [0.42, 0.45, 0.46, 0.38, 0.32, 0.28, 0.25, 0.23, 0.20],
}

def spectral_angle(image, reference_spectrum, bands):
    """
    Compute spectral angle (in degrees) between image pixel spectra and reference_spectrum.
      - image: ee.Image with bands selected in order 'bands'
      - reference_spectrum: python list of spectral values (same length as bands)
      - bands: list of band names (strings) in the same order
    """
    ref = ee.Image.constant(reference_spectrum).toArray()
    img = image.select(bands).toArray()
    dot = img.multiply(ref).arrayReduce(ee.Reducer.sum(), [0])
    norm_i = img.multiply(img).arrayReduce(ee.Reducer.sum(), [0]).sqrt()
    norm_r = ref.multiply(ref).arrayReduce(ee.Reducer.sum(), [0]).sqrt()
    cos_sim = dot.divide(norm_i.multiply(norm_r))

    cos_sim = cos_sim.max(-1).min(1)
    angle = cos_sim.acos().multiply(180 / 3.141592653589793)
    return angle.arrayProject([0]).arrayFlatten([['angle']])

sam_colors = {"Calcite":"red", "Kaolinite":"blue", "IronOxide":"gold"}

sam_images = []
for mineral, spec in references.items():
    angle = spectral_angle(asterDOS, spec, asterBandNames).rename(mineral + '_SAM')
    sam_images.append(angle)

    Map.addLayer(
        angle,
        {"min": 0, "max": 45, "palette": ['purple','white','orange','green','blue','red']},
        mineral + ' SAM Angle',
        shown=False
    )

    mask = angle.lt(8).selfMask().rename(mineral + '_Mask')
    Map.addLayer(mask, {"palette":[sam_colors[mineral]]}, mineral + ' SAM (Masked)', shown=False)


sam_image = ee.Image.cat(sam_images)
Map


Map(center=[7.962067105113433, 6.486600158075314], controls=(WidgetControl(options=['position', 'transparent_b…

In [22]:
#PCA: means, covariance, eigen-decomp, projection
bands = ['B01','B02','B3N','B04','B05','B06','B07','B08','B09']
image = asterDOS.select(bands).clip(kogi)  # DOS-corrected image

print("Band names:", image.bandNames().getInfo())

mean_dict = image.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=kogi,
    scale=30,
    maxPixels=1e13
)

band_names = ee.List(bands)
mean_list = band_names.map(lambda b: mean_dict.get(b))
mean_image = ee.Image.constant(mean_list).rename(band_names)

normalized = image.subtract(mean_image)

#Covariance matrix --
array_image = normalized.toArray()
covar_dict = array_image.reduceRegion(
    reducer=ee.Reducer.covariance(),
    geometry=kogi,
    scale=30,
    maxPixels=1e13
)

covar_array = ee.Array(covar_dict.get('array'))
print("Covariance matrix (server-side ee.Array). To view client-side, run covar_array.getInfo() below if desired.")


# Eigen decomposition --
eigens = covar_array.eigen()
eigenValues = eigens.slice(1, 0, 1)
eigenVectors = eigens.slice(1, 1)

flat_array = image.toArray().toArray(1)

projected = ee.Image(eigenVectors).matrixMultiply(flat_array)

pc_names = ['PC{}'.format(i) for i in range(1, len(bands)+1)]
pc_image = (projected
            .arrayProject([0])
            .arrayFlatten([pc_names]))
Map.addLayer(pc_image.select(['PC2','PC3','PC5']), {'min': -2, 'max': 2}, 'PCA RGB (PC2-PC3-PC5)', shown=False)

Map


Band names: ['B01', 'B02', 'B3N', 'B04', 'B05', 'B06', 'B07', 'B08', 'B09']
Covariance matrix (server-side ee.Array). To view client-side, run covar_array.getInfo() below if desired.


Map(bottom=1002344.0, center=[7.962067105113433, 6.486600158075314], controls=(WidgetControl(options=['positio…